# 1. Preparação

Nesta etapa, será feita a preparação dos dados, para que possamos dar andamento às tratativas para os dados subirem para a camada Gold.

In [1]:
# 1. Importação das bibliotecas e definição dos caminhos do projeto.
from pathlib import Path
import pandas as pd

RAIZ   = Path.cwd().parent
SILVER = RAIZ / "data" / "Silver"
GOLD   = RAIZ / "data" / "Gold"

# 2. Validação: Todos os arquivos da camada Silver precisam existir antes de continuar.
## Se algum estiver faltando, a mensagem de erro mostra qual é.
arquivos = ["dim_clientes", "dim_assinatura", "dim_planos", "dim_interacoes", "fato_pagamentos"]
faltando = [a for a in arquivos if not (SILVER / f"{a}.csv").exists()]
assert not faltando, f"Faltam arquivos na Silver: {faltando}. Rode os notebooks 01 a 06 antes do 07."

# 3. Criação da pasta da camada Gold, caso ela ainda não exista.
GOLD.mkdir(parents=True, exist_ok=True)

# 2. Leitura da Silver

Nesta etapa, será feita a leitura das tabelas tratadas na camada Silver.

In [2]:
# 1. Importação das tabelas da camada Silver.
dim_cliente     = pd.read_csv(SILVER / "dim_clientes.csv",    sep=";", parse_dates=["data_cadastro"])
dim_assinatura  = pd.read_csv(SILVER / "dim_assinatura.csv",  sep=";", parse_dates=["data_inicio", "data_cancelamento"])
dim_planos      = pd.read_csv(SILVER / "dim_planos.csv",      sep=";")
dim_interacoes  = pd.read_csv(SILVER / "dim_interacoes.csv",  sep=";", parse_dates=["data_interacao"])
fato_pagamentos = pd.read_csv(SILVER / "fato_pagamentos.csv", sep=";", parse_dates=["data_pagamento"])

# 2. Validação: Quantidade de linhas de cada tabela lida.
for nome, df in [("dim_cliente", dim_cliente), ("dim_assinatura", dim_assinatura), ("dim_planos", dim_planos),
                 ("dim_interacoes", dim_interacoes), ("fato_pagamentos", fato_pagamentos)]:
    print(f"{nome:16s} {len(df):6d} linhas")

dim_cliente        2001 linhas
dim_assinatura     3001 linhas
dim_planos            9 linhas
dim_interacoes    10000 linhas
fato_pagamentos   15000 linhas


# 3. Regras de Negócio

Nesta etapa, serão definidas as regras de negócio que se aplicam às tabelas que serão enviadas para a camada Gold.

In [3]:
# 1. Receita: Somente pagamentos com status "Pago" são considerados receita.
## A tabela de pagamentos também possui status de falha, pendência e atraso, que não representam dinheiro recebido.
STATUS_RECEITA  = "Pago"

# 2. Ordem dos status: Lista dos status válidos, na mesma ordem da legenda do gráfico no Google Sheets e da construção das tabelas.
ORDEM_STATUS    = ["Pago", "Pendente", "Atrasado", "Falhou"]

# 3. Tipos de interação: Lista dos tipos permitidos na tabela interacoes_por_tipo.
## Se aparecer um tipo fora da lista, o notebook é interrompido.
TIPOS_INTERACAO = ["Suporte", "Dúvida", "Reclamação", "Sugestão", "Cancelamento"]

# 4. Data de referência: Armazena a data mais recente dos dados e o mês dessa data.
## A data vem dos dados (e não do computador), para que rodar o notebook de novo gere o mesmo resultado.
DATA_REFERENCIA = fato_pagamentos["data_pagamento"].max()
MES_REFERENCIA  = DATA_REFERENCIA.to_period("M")

# 4.2 Mês incompleto: O mês de referência é incompleto quando a data de referência vem antes do último dia do mês.
MES_INCOMPLETO  = DATA_REFERENCIA.normalize() < MES_REFERENCIA.end_time.normalize()

# 5. Cancelamentos: Cancelamento EFETIVO é aquele com data_cancelamento <= DATA_REFERENCIA.
## Data futura = cancelamento AGENDADO (a assinatura ainda está ativa).

# 6. Validação: Mostra o resultado das variáveis armazenadas.
print("Data de referência:", DATA_REFERENCIA.date(), "| mês incompleto?", MES_INCOMPLETO)

Data de referência: 2026-09-06 | mês incompleto? True


# 4. Criação de Tabelas

Nesta etapa, será feita a criação de todas as tabelas da camada Gold. Os resultados dessas tabelas serão armazenados no Google Planilhas, para a construção do dashboard. Com esse resumo, é possível reduzir a quantidade de linhas e já levar os dados tratados para a visualização do dashboard.

## 4.1 Criação da Tabela receita_mensal (fato_pagamentos)

Nesta etapa, é construída a tabela Gold com a receita paga por mês. Somente pagamentos com status Pago entram na receita. Cada linha da tabela representa 1 mês. Esta tabela alimenta o gráfico "Receita por mês" do dashboard.

In [4]:
# 1. Filtro: Apenas os pagamentos com status "Pago" são considerados receita, conforme a regra de negócio.
pagos = fato_pagamentos[fato_pagamentos["status_pagamento"] == STATUS_RECEITA].copy()

# 1.2 Criação da coluna mes (ex.: 2024-05) a partir da data do pagamento, para agrupar por período.
pagos["mes"] = pagos["data_pagamento"].dt.to_period("M").astype(str)

# 1.3 Agrupamento: Para reduzir a quantidade de linhas, os pagamentos são agrupados por mês.
## São calculadas a quantidade de pagamentos, a soma da receita paga (valor tratado) e a soma da receita original (valor com números negativos).
receita_mensal = (
    pagos.groupby("mes", as_index=False)                        # Agrupamento por mês
         .agg(
             qtd_pagamentos=("id_pagamento", "count"),          # Quantidade de pagamentos
             receita_paga=("valor", "sum"),                     # Soma dos valores dos pagamentos "Pago" (coluna de valor tratada)
             receita_paga_original=("valor_original", "sum"),   # Soma dos valores dos pagamentos "Pago" (coluna original, que possui valores negativos)
         )
         .round(2)                                              # Arredondamento em centavos, para evitar pequenas sobras de cálculo com decimais
)

# 1.4 Criação da coluna ticket_medio: receita paga dividida pela quantidade de pagamentos.
receita_mensal["ticket_medio"] = (receita_mensal["receita_paga"] / receita_mensal["qtd_pagamentos"]).round(2)

# 1.5 Criação da coluna mes_incompleto: True apenas no mês de referência, quando ele ainda não terminou.
receita_mensal["mes_incompleto"] = (receita_mensal["mes"] == str(MES_REFERENCIA)) & MES_INCOMPLETO

## 4.2 Criação da Tabela pagamentos_status_mensal (fato_pagamentos)

Nesta etapa, é construída a tabela Gold com a proporção de cada status de pagamento (Pago, Pendente, Atrasado e Falhou) em cada mês. Cada linha representa 1 mês e 1 status. Esta tabela alimenta o gráfico "Status dos pagamentos" do dashboard.

In [5]:
# 1. Validação pré-tratamento: Se aparecer um status que não está na lista, o notebook para.
desconhecidos = set(fato_pagamentos["status_pagamento"]) - set(ORDEM_STATUS)
assert not desconhecidos, f"Status novo(s) fora da lista: {desconhecidos}"

pagamentos_status = fato_pagamentos.copy()
pagamentos_status["mes"] = pagamentos_status["data_pagamento"].dt.to_period("M").astype(str)

# 2. Agrupamento de dados: Por mês e status, com a quantidade de pagamentos e a soma de valores.
pagamentos_status_mensal = (
    pagamentos_status.groupby(["mes", "status_pagamento"], as_index=False)
                     .agg(
                         qtd_pagamentos=("id_pagamento", "count"),
                         valor_total=("valor", "sum"),
                     )
)

# 2.2 Total de pagamentos de cada mês, repetido em todas as linhas daquele mês (base para o percentual).
total_mes = pagamentos_status_mensal.groupby("mes")["qtd_pagamentos"].transform("sum")

# 2.3 Criação de novas colunas: percentual de cada status dentro do mês (pela quantidade), valor arredondado e aviso de mês incompleto.
pagamentos_status_mensal["pct_do_mes"]     = (pagamentos_status_mensal["qtd_pagamentos"] / total_mes).round(4)
pagamentos_status_mensal["valor_total"]    = pagamentos_status_mensal["valor_total"].round(2)
pagamentos_status_mensal["mes_incompleto"] = (pagamentos_status_mensal["mes"] == str(MES_REFERENCIA)) & MES_INCOMPLETO

# 2.4 Ordenação: Por mês e, dentro do mês, na ordem de ORDEM_STATUS.
ordem = {status: i for i, status in enumerate(ORDEM_STATUS)}
pagamentos_status_mensal["_ordem"] = pagamentos_status_mensal["status_pagamento"].map(ordem)
pagamentos_status_mensal = (pagamentos_status_mensal.sort_values(["mes", "_ordem"])
                                                    .drop(columns="_ordem")
                                                    .reset_index(drop=True))

## 4.3 Criação da Tabela assinaturas_mensal (dim_assinatura)

Nesta etapa, é construída a tabela Gold com as assinaturas novas, os cancelamentos e as assinaturas ativas ao final de cada mês. Cada linha representa 1 mês. Por regra de negócio, o cancelamento com data futura é considerado agendado, e a assinatura continua ativa até a data chegar. Esta tabela alimenta os gráficos "Assinaturas ativas" e "Novas e canceladas" do dashboard.

In [6]:
# 1. Preparação: DataFrame apenas com assinaturas reais (sem o registro -1, criado no tratamento dos órfãos), a partir de uma cópia da tabela Silver já tratada.
assinaturas = dim_assinatura[dim_assinatura["id_assinatura"] != -1].copy()

# 1.2 Definição dos meses (do primeiro início de assinatura até o mês de referência) para o agrupamento.
## Os cancelamentos são separados em efetivos (a data já chegou) e agendados (a data ainda vai chegar).
meses = pd.period_range(assinaturas["data_inicio"].min().to_period("M"), MES_REFERENCIA, freq="M")
cancel_efetivos = assinaturas[assinaturas["data_cancelamento"] <= DATA_REFERENCIA]
cancelamentos_agendados = int((assinaturas["data_cancelamento"] > DATA_REFERENCIA).sum())

# 1.3 Contagem por mês: Transforma cada data em mês (ex.: 2024-05-19 vira 2024-05) e conta quantas assinaturas começaram e quantas foram canceladas.
## O reindex garante que todos os meses apareçam, com 0 onde não houve nada.
novas         = assinaturas["data_inicio"].dt.to_period("M").value_counts().reindex(meses, fill_value=0)
cancelamentos = cancel_efetivos["data_cancelamento"].dt.to_period("M").value_counts().reindex(meses, fill_value=0)

# 1.4 Assinaturas ativas ao final de cada mês: novas assinaturas acumuladas - cancelamentos acumulados.
ativas_fim    = novas.cumsum() - cancelamentos.cumsum()
# 1.5 Assinaturas ativas no início de cada mês: é o valor do fim do mês anterior (o shift(1) empurra os valores um mês para baixo).
ativas_inicio = ativas_fim.shift(1, fill_value=0)
# 1.6 Taxa de cancelamento do mês: cancelamentos / ativas no início do mês. O .where(> 0) evita divisão por zero no primeiro mês.
taxa          = (cancelamentos / ativas_inicio.where(ativas_inicio > 0)).round(4)

# 1.7 Criação da tabela para o Google Sheets.
## Cada item do dicionário vira uma coluna (ex.: mes, novas_assinaturas), e o conteúdo vem das variáveis criadas acima.
assinaturas_mensal = pd.DataFrame({
    "mes": meses.astype(str),
    "novas_assinaturas": novas.values,
    "cancelamentos": cancelamentos.values,
    "assinaturas_ativas_fim_mes": ativas_fim.values,
    "taxa_cancelamento": taxa.values,
})

assinaturas_mensal["mes_incompleto"] = (assinaturas_mensal["mes"] == str(MES_REFERENCIA)) & MES_INCOMPLETO

# 1.8 Meses depois da última data de início de assinatura = "sem dados de novas assinaturas" (vazio, e não zero).
ultimo_mes_inicio = str(assinaturas["data_inicio"].max().to_period("M"))
# 1.9 Conversão da coluna para um tipo de número inteiro que aceita vazio. Sem isso, o vazio faria a coluna inteira virar decimal, e 124 apareceria como 124.0.
assinaturas_mensal["novas_assinaturas"] = assinaturas_mensal["novas_assinaturas"].astype("Int64")
# 2. Nas linhas em que o mês é depois do último mês com início de assinatura (hoje, 2026-04), novas_assinaturas fica vazio.
## A comparação funciona entre textos, porque "2026-05" vem depois de "2026-04" na ordem alfabética.
## Assim, os meses seguintes ficam vazios e não zerados, e o dashboard não passa a ideia de que a empresa parou de vender.
assinaturas_mensal.loc[assinaturas_mensal["mes"] > ultimo_mes_inicio, "novas_assinaturas"] = pd.NA

# 3. Validação visual: Mostra os 4 primeiros meses passo a passo (novas, cancelamentos, ativas e taxa) para conferir os cálculos.
pd.DataFrame({"novas": novas, "canc": cancelamentos, "ativas_fim": ativas_fim,
              "ativas_inicio": ativas_inicio, "taxa": taxa}).head(4)

,novas,canc,ativas_fim,ativas_inicio,taxa
2024-01,124,0,124,0,NaN
2024-02,107,0,231,124,0.0000
2024-03,105,4,332,231,0.0173
2024-04,103,5,430,332,0.0151


## 4.4 Criação da Tabela assinaturas_por_plano (dim_assinatura, dim_planos e fato_pagamentos)

Nesta etapa, é construída a tabela Gold que resume as assinaturas de cada plano: quantas existem, quantas estão ativas, canceladas ou agendadas, o percentual de cancelamento e a receita paga. Cada linha representa 1 plano. Esta tabela alimenta o gráfico "Cancelamento por plano" do dashboard.

In [7]:
# 1. Preparação: Cópia das assinaturas reais (sem o registro -1) para criar colunas auxiliares sem alterar a tabela original.
ass_plano = assinaturas.copy()

# 1.2 Classificação: Cancelamento EFETIVO é aquele cuja data já chegou (até a data de referência).
## Assinaturas sem data de cancelamento retornam False, pois uma data vazia não passa na comparação.
ass_plano["cancelada"] = ass_plano["data_cancelamento"] <= DATA_REFERENCIA

# 1.3 Classificação: Cancelamento AGENDADO é aquele com data futura. A assinatura ainda está ativa.
ass_plano["agendada"]  = ass_plano["data_cancelamento"] >  DATA_REFERENCIA

# 1.4 Agrupamento: Conta, para cada plano, quantas assinaturas existem, quantas foram canceladas e quantas estão agendadas.
## O .sum() em uma coluna True/False conta quantos valores são True.
por_plano = (
    ass_plano.groupby("id_plano", as_index=False)         # um grupo por plano (as_index=False mantém id_plano como coluna)
             .agg(
                 qtd_assinaturas=("id_assinatura", "count"),  # total de assinaturas do plano
                 canceladas=("cancelada", "sum"),             # cancelamentos efetivos
                 agendadas=("agendada", "sum"),               # cancelamentos agendados (data futura)
             )
)

# 1.5 Junção (merge): Traz os dados do plano (nome, categoria e preço) para a tabela de contagens.
## how="left" mantém TODOS os planos da dim_planos, mesmo os que não possuem assinaturas.
assinaturas_por_plano = dim_planos[["id_plano", "nome_plano", "categoria", "valor_mensal"]].merge(
    por_plano, on="id_plano", how="left"
)

# 1.6 Tratamento: Planos sem assinaturas ficam com valores vazios após o merge.
## Os vazios são trocados por 0 e as colunas voltam a ser números inteiros.
colunas_contagem = ["qtd_assinaturas", "canceladas", "agendadas"]
assinaturas_por_plano[colunas_contagem] = assinaturas_por_plano[colunas_contagem].fillna(0).astype(int)

# 2. Cálculo: Assinaturas ativas = total de assinaturas - canceladas efetivas.
## Os cancelamentos agendados continuam contando como ativas.
assinaturas_por_plano["ativas"] = assinaturas_por_plano["qtd_assinaturas"] - assinaturas_por_plano["canceladas"]

# 3. Cálculo: Percentual de cancelamento = canceladas / total de assinaturas (em fração, ex.: 0.2908).
## O .where(> 0) troca zeros por vazio, evitando divisão por zero em planos sem assinaturas.
assinaturas_por_plano["pct_cancelamento"] = (
    assinaturas_por_plano["canceladas"] / assinaturas_por_plano["qtd_assinaturas"].where(assinaturas_por_plano["qtd_assinaturas"] > 0)
).round(4)

# 4. Junção (merge): Descobre o plano de cada pagamento pago, seguindo o caminho pagamento -> assinatura -> plano.
## Usa a dim_assinatura completa (com o registro -1), para que pagamentos de assinatura "não identificada" não sejam perdidos.
pagos_plano = pagos[["id_assinatura", "valor"]].merge(
    dim_assinatura[["id_assinatura", "id_plano"]], on="id_assinatura", how="left"
)

# 4.1 Agrupamento: Soma o valor dos pagamentos de cada plano.
receita_plano = pagos_plano.groupby("id_plano", as_index=False).agg(receita_paga=("valor", "sum"))

# 4.2 Junção (merge): Traz a receita de cada plano para a tabela principal.
assinaturas_por_plano = assinaturas_por_plano.merge(receita_plano, on="id_plano", how="left")

# 4.3 Tratamento: Plano sem pagamentos fica com receita 0 e os valores são arredondados em centavos.
assinaturas_por_plano["receita_paga"] = assinaturas_por_plano["receita_paga"].fillna(0).round(2)

## 4.5 Criação da Tabela interacoes_por_tipo (dim_interacoes)

Nesta etapa, é construída a tabela Gold que resume as interações de atendimento por tipo (Suporte, Dúvida, Reclamação, Sugestão e Cancelamento). Cada linha representa 1 tipo. Esta tabela alimenta o gráfico "Atendimento por tipo" do dashboard.

In [8]:
# 1. Validação pré-tratamento: Se existir um tipo fora da lista esperada (ex.: "Investigar"), o notebook é interrompido.
tipos_desconhecidos = set(dim_interacoes["tipo"]) - set(TIPOS_INTERACAO)
assert not tipos_desconhecidos, f"Tipo(s) de interação fora da lista: {tipos_desconhecidos}"

# 2. Agrupamento: Conta quantas interações existem em cada tipo (sem nenhum filtro: todas as interações entram).
interacoes_por_tipo = (
    dim_interacoes.groupby("tipo", as_index=False)
                  .agg(qtd_interacoes=("id_interacao", "count"))
)

# 3. Cálculo: Percentual de cada tipo em relação ao total de interações (em fração, ex.: 0.3567).
interacoes_por_tipo["pct_do_total"] = (interacoes_por_tipo["qtd_interacoes"] / interacoes_por_tipo["qtd_interacoes"].sum()).round(4)

# 4. Ordenação: Do tipo com mais interações para o com menos (mesma ordem do gráfico do dashboard).
interacoes_por_tipo = (interacoes_por_tipo.sort_values(["qtd_interacoes", "tipo"], ascending=[False, True])
                                          .reset_index(drop=True))

## 4.6 Criação da Tabela metadados

Nesta etapa, é construída a tabela Gold que descreve os próprios dados: o período que eles cobrem, a data de referência, os avisos sobre limitações e a data em que a camada Gold foi gerada. A tabela possui apenas 1 linha.

Esta tabela alimenta os textos informativos do dashboard, como "Dados até 06/09/2026", o período do topo, o último ponto tracejado dos gráficos e o aviso sobre as novas assinaturas. As tabelas `receita_mensal` e `assinaturas_mensal` vêm das seções anteriores.

In [9]:
# 1. Descoberta: Último mês em que existem novas assinaturas (os meses seguintes estão vazios na assinaturas_mensal).
ultimo_mes_novas = assinaturas_mensal.loc[assinaturas_mensal["novas_assinaturas"].notna(), "mes"].max()

# 2. Construção: Tabela de 1 linha que descreve os dados (período, data de referência, avisos e data de geração).
## As datas são gravadas como texto no formato ISO (aaaa-mm-dd), para não depender do formato do computador.
metadados = pd.DataFrame([{
    "mes_inicial": receita_mensal["mes"].min(),                          # primeiro mês das tabelas mensais
    "mes_final": receita_mensal["mes"].max(),                            # último mês das tabelas mensais
    "data_referencia": DATA_REFERENCIA.strftime("%Y-%m-%d"),             # última data presente nos dados
    "mes_incompleto": bool(MES_INCOMPLETO),                              # True se o último mês tem dados parciais
    "ultimo_mes_novas_assinaturas": ultimo_mes_novas,                    # depois desse mês não há dados de novas assinaturas
    "data_geracao": pd.Timestamp.today().strftime("%Y-%m-%d"),           # dia em que a Gold foi gerada (muda a cada execução)
}])

## 4.7 Criação da Tabela kpis_resumo

Nesta etapa, é construída a tabela Gold com os indicadores gerais da empresa, que alimentam os cartões do topo do dashboard. A tabela possui apenas 1 linha.

Os indicadores não são calculados novamente a partir da camada Silver: eles são extraídos das tabelas Gold já construídas (`receita_mensal`, `pagamentos_status_mensal`, `assinaturas_por_plano` e `interacoes_por_tipo`). Dessa forma, os cartões do topo e os gráficos sempre mostram os mesmos números.

In [10]:
# 1. Extração: Cada indicador é tirado das tabelas Gold já construídas (e não direto da Silver),
## para que os cartões do topo mostrem exatamente os mesmos números dos gráficos.

# 1.1 Pagamentos: receita e quantidade vêm da receita_mensal; o total de pagamentos vem da pagamentos_status_mensal.
receita_total        = round(float(receita_mensal["receita_paga"].sum()), 2)
qtd_pagamentos_pagos = int(receita_mensal["qtd_pagamentos"].sum())
qtd_pagamentos_total = int(pagamentos_status_mensal["qtd_pagamentos"].sum())

# 1.2 Assinaturas: os totais vêm da assinaturas_por_plano (soma de todos os planos).
assinaturas_total       = int(assinaturas_por_plano["qtd_assinaturas"].sum())
assinaturas_ativas      = int(assinaturas_por_plano["ativas"].sum())
assinaturas_canceladas  = int(assinaturas_por_plano["canceladas"].sum())
cancelamentos_agendados = int(assinaturas_por_plano["agendadas"].sum())

# 1.3 Atendimento: o total vem da interacoes_por_tipo.
total_interacoes = int(interacoes_por_tipo["qtd_interacoes"].sum())

# 2. Construção: Tabela de 1 linha, com os indicadores dos cartões e os números usados nas legendas.
kpis_resumo = pd.DataFrame([{
    "receita_total": receita_total,
    "ticket_medio": round(receita_total / qtd_pagamentos_pagos, 2),
    "assinaturas_ativas": assinaturas_ativas,
    "cancelamentos_agendados": cancelamentos_agendados,
    "pct_cancelamento": round(assinaturas_canceladas / assinaturas_total, 4),
    "assinaturas_canceladas": assinaturas_canceladas,
    "assinaturas_total": assinaturas_total,
    "pct_pagamentos_pagos": round(qtd_pagamentos_pagos / qtd_pagamentos_total, 4),
    "qtd_pagamentos_pagos": qtd_pagamentos_pagos,
    "qtd_pagamentos_total": qtd_pagamentos_total,
    "total_interacoes": total_interacoes,
}])

# 5. Conferência de Dados

Nesta etapa, cada tabela é conferida contra a camada Silver e contra as outras tabelas Gold. Nenhum arquivo é gravado nesta etapa: se qualquer conferência falhar, o notebook é interrompido com um erro e a camada Gold permanece intacta, exigindo a correção antes de avançar.

## 5.1 Conferência da receita_mensal vs fato_pagamentos

Nesta etapa, o total e a quantidade de pagamentos pagos da receita_mensal são conferidos contra a Silver. Também é conferido que não existe mês repetido e que apenas o último mês pode estar incompleto.

In [11]:
# 1. Totais para comparação: A receita da tabela Gold (receita_mensal) e a receita da camada Silver (apenas pagamentos "Pago").
total_gold   = round(receita_mensal["receita_paga"].sum(), 2)
total_silver = round(pagos["valor"].sum(), 2)

# 1.2 Validação entre Gold e Silver: Verifica se os dados continuam corretos após o tratamento do notebook 07_camada_gold.
assert total_gold == total_silver, f"Total não bate: Gold={total_gold} Silver={total_silver}"
assert receita_mensal["qtd_pagamentos"].sum() == len(pagos), "Quantidade de pagamentos não bate."
assert receita_mensal["mes"].is_unique, "Existe mês repetido."
assert receita_mensal["mes_incompleto"].sum() <= 1, "Só o último mês pode ser incompleto."

# 1.3 Se todos os testes forem aprovados, será exibida a mensagem abaixo.
print("Conciliação OK")

Conciliação OK


## 5.2 Conferência da pagamentos_status_mensal vs fato_pagamentos

Nesta etapa, a quantidade e o valor de todos os pagamentos são conferidos contra a Silver. Também é conferido que os percentuais de cada mês somam 100%, que não existe mês e status repetidos, que todo mês tem os 4 status e que os pagamentos pagos batem com a receita_mensal.

In [12]:
# 1. Conferência dos dados que irão para a camada Gold com a Silver (Em caso de divergência, o notebook é interrompido e nenhum arquivo é gravado.)
assert pagamentos_status_mensal["qtd_pagamentos"].sum() == len(fato_pagamentos), "Quantidade não bate com a Silver."
assert abs(pagamentos_status_mensal["valor_total"].sum() - fato_pagamentos["valor"].sum()) < 0.01, "Valor total não bate com a Silver."

assert (pagamentos_status_mensal.groupby("mes")["pct_do_mes"].sum().round(2) == 1).all(), "Percentuais de algum mês não somam 100%."
assert not pagamentos_status_mensal.duplicated(["mes", "status_pagamento"]).any(), "Existe mês + status repetido."
assert (pagamentos_status_mensal.groupby("mes").size() == len(ORDEM_STATUS)).all(), "Algum mês não tem os 4 status."

so_pagos = pagamentos_status_mensal[pagamentos_status_mensal["status_pagamento"] == STATUS_RECEITA]
assert so_pagos["qtd_pagamentos"].sum() == receita_mensal["qtd_pagamentos"].sum(), "Pagos: quantidade difere da receita_mensal."
assert abs(so_pagos["valor_total"].sum() - receita_mensal["receita_paga"].sum()) < 0.01, "Pagos: valor difere da receita_mensal."
print("Conciliação OK")

Conciliação OK


## 5.3 Conferência da assinaturas_mensal vs dim_assinatura

Nesta etapa, as novas assinaturas, os cancelamentos e as ativas do último mês são conferidos contra a Silver. Também é conferido que as ativas nunca ficam negativas, que cancelamentos efetivos mais agendados fecham o total com data de cancelamento e que nenhum cancelamento acontece antes do início.

In [13]:
# 1. Conferência dos dados que irão para a camada Gold com a Silver (Em caso de divergência, o notebook é interrompido e nenhum arquivo é gravado.)
assert assinaturas_mensal["novas_assinaturas"].sum() == len(assinaturas), "Novas assinaturas não batem com a Silver."
assert assinaturas_mensal["cancelamentos"].sum() == len(cancel_efetivos), "Cancelamentos não batem com a Silver."
assert assinaturas_mensal["assinaturas_ativas_fim_mes"].iloc[-1] == len(assinaturas) - len(cancel_efetivos), "Ativas do último mês não batem."
assert (assinaturas_mensal["assinaturas_ativas_fim_mes"] >= 0).all(), "Existe mês com ativas negativas."
assert assinaturas_mensal["mes"].is_unique, "Existe mês repetido."
assert len(cancel_efetivos) + cancelamentos_agendados == assinaturas["data_cancelamento"].notna().sum(), "Cancelamentos efetivos + agendados não fecham."
assert (assinaturas["data_cancelamento"] < assinaturas["data_inicio"]).sum() == 0, "Há cancelamento antes do início."
print("Conciliação OK | cancelamentos agendados (não contados):", cancelamentos_agendados)

Conciliação OK | cancelamentos agendados (não contados): 139


## 5.4 Conferência da assinaturas_por_plano vs dim_assinatura

Nesta etapa, é conferido que os merges não multiplicam nem perdem pagamentos, que todos os planos aparecem uma única vez, que os totais batem com a Silver e que a receita somada por plano é igual à receita total.

In [14]:
# 1. Validação dos merges: Os pagamentos não podem ser multiplicados nem perdidos, e todos precisam ter um plano.
assert len(pagos_plano) == len(pagos), "O merge multiplicou ou perdeu pagamentos."
assert pagos_plano["id_plano"].notna().all(), "Existe pagamento sem plano."

# 2. Validação dos planos: Todos os planos da dim_planos devem aparecer, sem repetição.
assert len(assinaturas_por_plano) == len(dim_planos), "Algum plano sumiu ou foi duplicado."
assert assinaturas_por_plano["nome_plano"].is_unique, "Existe plano repetido."

# 3. Conciliação com a Silver: Os totais da tabela Gold precisam bater com a origem.
assert assinaturas_por_plano["qtd_assinaturas"].sum() == len(assinaturas), "Assinaturas não batem com a Silver."
assert assinaturas_por_plano["canceladas"].sum() == len(cancel_efetivos), "Canceladas não batem com a Silver."
assert assinaturas_por_plano["agendadas"].sum() == cancelamentos_agendados, "Agendadas não batem."
assert assinaturas_por_plano["ativas"].sum() == len(assinaturas) - len(cancel_efetivos), "Ativas não batem."

# 4. Conciliação com a receita total: A soma por plano precisa ser igual à soma de todos os pagamentos pagos.
## A margem de 0.01 (1 centavo) evita falhas por pequenas sobras do cálculo com decimais.
assert abs(assinaturas_por_plano["receita_paga"].sum() - pagos["valor"].sum()) < 0.01, "Receita por plano não bate com a receita total."
print("Conciliação OK")

Conciliação OK


## 5.5 Conferência da interacoes_por_tipo vs dim_interacoes

Nesta etapa, é conferido que a soma das interações por tipo é igual ao total da Silver, que todos os tipos esperados aparecem e que os percentuais somam 100%.

In [15]:
# 1. Conciliação com a Silver: A soma das quantidades precisa ser igual ao total de linhas da tabela de interações.
assert interacoes_por_tipo["qtd_interacoes"].sum() == len(dim_interacoes), "Quantidade não bate com a Silver."

# 2. Validação da forma: Um tipo por linha, todos os tipos esperados presentes e percentuais somando 100%.
assert interacoes_por_tipo["tipo"].is_unique, "Existe tipo repetido."
assert set(interacoes_por_tipo["tipo"]) == set(TIPOS_INTERACAO), "Algum tipo esperado não apareceu."
assert abs(interacoes_por_tipo["pct_do_total"].sum() - 1) < 0.001, "Percentuais não somam 100%."
print("Conciliação OK")

Conciliação OK


## 5.6 Conferência da metadados

Nesta etapa, é conferido que o período, a data de referência, o aviso de mês incompleto e o último mês de novas assinaturas estão coerentes com as tabelas mensais, e que a Gold não foi gerada antes da data dos dados.

In [16]:
# 1. Validação da forma: A tabela precisa ter exatamente 1 linha.
assert len(metadados) == 1, "A tabela metadados precisa ter 1 linha."

# 2. Conciliação com as tabelas Gold: Os avisos do dashboard não podem contradizer os gráficos.
## 2.1 Período: o mesmo início e o mesmo fim nas tabelas mensais.
assert metadados["mes_inicial"].iloc[0] == receita_mensal["mes"].min() == assinaturas_mensal["mes"].min(), "Mês inicial difere entre as tabelas."
assert metadados["mes_final"].iloc[0] == receita_mensal["mes"].max() == assinaturas_mensal["mes"].max(), "Mês final difere entre as tabelas."

## 2.2 Data de referência: precisa estar dentro do último mês.
assert metadados["data_referencia"].iloc[0][:7] == metadados["mes_final"].iloc[0], "A data de referência não está no último mês."

## 2.3 Mês incompleto: o mesmo aviso nas duas tabelas.
assert metadados["mes_incompleto"].iloc[0] == receita_mensal["mes_incompleto"].any() == assinaturas_mensal["mes_incompleto"].any(), "Aviso de mês incompleto difere."

## 2.4 Novas assinaturas: depois do último mês informado, todos os meses precisam estar vazios.
assert assinaturas_mensal.loc[assinaturas_mensal["mes"] > ultimo_mes_novas, "novas_assinaturas"].isna().all(), "Existem novas assinaturas depois do último mês informado."

# 3. Validação de datas: A Gold não pode ter sido gerada antes da data dos dados (datas ISO comparam como texto).
assert metadados["data_geracao"].iloc[0] >= metadados["data_referencia"].iloc[0], "Data de geração anterior à data de referência."
print("Conciliação OK")

Conciliação OK


## 5.7 Conferência da kpis_resumo

Nesta etapa, é conferido que cada cartão do topo bate com a Silver e com as outras tabelas Gold, para que os cartões e os gráficos mostrem sempre os mesmos números.

In [17]:
# 1. Validação da forma: A tabela precisa ter exatamente 1 linha.
assert len(kpis_resumo) == 1, "A tabela kpis_resumo precisa ter 1 linha."

# 2. Conciliação com a Silver: Os totais de origem precisam bater com os cartões.
assert qtd_pagamentos_total == len(fato_pagamentos), "Total de pagamentos não bate com a Silver."
assert assinaturas_total == len(assinaturas), "Total de assinaturas não bate com a Silver."
assert total_interacoes == len(dim_interacoes), "Total de interações não bate com a Silver."
assert abs(receita_total - pagos["valor"].sum()) < 0.01, "Receita total não bate com a Silver."

# 3. Conciliação entre tabelas Gold: Cada cartão precisa contar a mesma história dos gráficos.
## 3.1 Pagos: a quantidade da receita_mensal é a mesma da pagamentos_status_mensal.
qtd_pagos_status = pagamentos_status_mensal.loc[pagamentos_status_mensal["status_pagamento"] == STATUS_RECEITA, "qtd_pagamentos"].sum()
assert qtd_pagamentos_pagos == qtd_pagos_status, "Pagos: receita_mensal e pagamentos_status_mensal divergem."

## 3.2 Receita: o total por mês é o mesmo total por plano.
assert abs(receita_total - assinaturas_por_plano["receita_paga"].sum()) < 0.01, "Receita: meses e planos divergem."

## 3.3 Assinaturas: o cartão bate com o último mês e com os totais da assinaturas_mensal.
assert assinaturas_ativas == assinaturas_mensal["assinaturas_ativas_fim_mes"].iloc[-1], "Ativas: cartão difere do último mês."
assert assinaturas_canceladas == assinaturas_mensal["cancelamentos"].sum(), "Canceladas: cartão difere da assinaturas_mensal."
assert assinaturas_total == assinaturas_mensal["novas_assinaturas"].sum(), "Total: cartão difere das novas assinaturas."
assert assinaturas_ativas + assinaturas_canceladas == assinaturas_total, "Ativas + canceladas não fecham o total."

# 4. Validação de percentuais: Precisam estar entre 0 e 1.
assert kpis_resumo[["pct_cancelamento", "pct_pagamentos_pagos"]].iloc[0].between(0, 1).all(), "Percentual fora do intervalo de 0 a 1."
print("Conciliação OK")

Conciliação OK


# 6. Gravação na camada Gold

Nesta etapa, as 7 tabelas são gravadas na camada Gold de uma só vez. Esta etapa só é executada se todas as conferências da seção 5 foram aprovadas: um erro em qualquer conferência interrompe o notebook antes daqui, e nenhum arquivo da Gold é alterado. Assim, a camada Gold nunca fica com tabelas de execuções diferentes misturadas.

In [18]:
# 1. Gravação: Só chega aqui se TODAS as conferências da seção 5 foram aprovadas.
## Um erro em qualquer conferência interrompe o notebook antes deste ponto, e nenhum arquivo da Gold é alterado.
tabelas_gold = {
    "receita_mensal": receita_mensal,
    "pagamentos_status_mensal": pagamentos_status_mensal,
    "assinaturas_mensal": assinaturas_mensal,
    "assinaturas_por_plano": assinaturas_por_plano,
    "interacoes_por_tipo": interacoes_por_tipo,
    "metadados": metadados,
    "kpis_resumo": kpis_resumo,
}

# 1.2 Cada tabela é gravada com o mesmo padrão: separador ";", decimal "," e codificação utf-8-sig.
for nome, tabela in tabelas_gold.items():
    tabela.to_csv(GOLD / f"{nome}.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
    print(f"Gold gerada: {nome:26s} {len(tabela):4d} linhas")

Gold gerada: receita_mensal               33 linhas
Gold gerada: pagamentos_status_mensal    132 linhas
Gold gerada: assinaturas_mensal           33 linhas
Gold gerada: assinaturas_por_plano         9 linhas
Gold gerada: interacoes_por_tipo           5 linhas
Gold gerada: metadados                     1 linhas
Gold gerada: kpis_resumo                   1 linhas
